# Notebook 02 — Analisis Statistik Deskriptif & EDA
**Input**: `data/processed/demografi_clean.csv`  
**Tujuan**: Eksplorasi mendalam distribusi penduduk, tren pertumbuhan, uji hipotesis, dan analisis ketimpangan.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)

df = pd.read_csv('../data/processed/demografi_clean.csv')
print(f'Data loaded: {df.shape}')
df.head(3)

Data loaded: (38, 27)


,id,provinsi,jumlah_penduduk_ribu_2026,laju_pertumbuhan_pct_2026,distribusi_pct_2026,kepadatan_per_km2_2026,rasio_jenis_kelamin_2026,kepadatan_per_km2_2021,laju_1971_1980,laju_1980_1990,...,laju_2020_2021,laju_2020_2022,laju_2020_2023,laju_2020_2024,u5mr_per1000_2020,cbr_per1000_2020,tfr_2020,imr_per1000_2020,catatan,pulau
0,1,ACEH,5695.9000,1.3400,1.9800,100.0000,100.9000,92.0000,2.9300,2.7200,...,1.4900,1.4300,1.4100,1.3900,22.8800,19.6400,2.4200,19.4100,NaN,Sumatera
1,2,BALI,4488.2000,0.6800,1.5600,804.0000,100.2000,755.0000,1.6900,1.1800,...,1.4000,1.2900,0.7300,0.7100,15.3700,15.4200,2.0400,13.2600,NaN,Lainnya
2,3,BANTEN,12641.3000,1.0500,4.4000,1351.0000,103.1000,1248.0000,NaN,NaN,...,1.7600,1.6600,1.2200,1.1600,16.1500,16.5100,2.0100,13.8300,NaN,Jawa


## 1. Statistik Deskriptif Lengkap

In [2]:
target_cols = [
    'jumlah_penduduk_ribu_2026', 'kepadatan_per_km2_2026', 'laju_pertumbuhan_pct_2026',
    'laju_2020_2024', 'imr_per1000_2020', 'cbr_per1000_2020', 'tfr_2020', 'u5mr_per1000_2020'
]

def extended_describe(series):
    s = series.dropna()
    return pd.Series({
        'n'         : len(s),
        'mean'      : s.mean(),
        'median'    : s.median(),
        'std'       : s.std(),
        'cv (%)'    : s.std() / s.mean() * 100,
        'min'       : s.min(),
        'max'       : s.max(),
        'skewness'  : stats.skew(s),
        'kurtosis'  : stats.kurtosis(s),
        'sw_stat'   : stats.shapiro(s)[0],
        'sw_pval'   : stats.shapiro(s)[1]
    })

summary = pd.DataFrame({col: extended_describe(df[col]) for col in target_cols})
print('=== STATISTIK DESKRIPTIF EXTENDED ===')
summary.T

=== STATISTIK DESKRIPTIF EXTENDED ===


,n,mean,median,std,cv (%),min,max,skewness,kurtosis,sw_stat,sw_pval
jumlah_penduduk_ribu_2026,38.0000,7557.8526,3807.2000,11500.9096,152.1717,557.2000,51163.9000,2.7291,6.5762,0.5756,0.0000
kepadatan_per_km2_2026,38.0000,682.0000,102.0000,2603.7403,381.7801,5.0000,16129.0000,5.7103,31.3875,0.2428,0.0000
laju_pertumbuhan_pct_2026,38.0000,1.2758,1.3150,0.4278,33.5288,0.1800,3.0600,1.2943,7.0537,0.8158,0.0000
laju_2020_2024,38.0000,1.2958,1.3650,0.3206,24.7433,0.3100,1.9300,-1.0237,1.3254,0.9207,0.0103
imr_per1000_2020,33.0000,19.7945,17.2300,7.1658,36.2009,10.3800,38.1700,1.0202,0.2276,0.8846,0.0022
cbr_per1000_2020,32.0000,18.4516,18.8350,2.3252,12.6017,13.6900,22.8400,-0.2604,-0.4543,0.9735,0.6019
tfr_2020,32.0000,2.3034,2.3050,0.2398,10.4108,1.7500,2.7900,-0.0547,-0.1472,0.9894,0.9849
u5mr_per1000_2020,34.0000,23.5841,20.2500,9.2870,39.3782,12.0200,49.0400,1.2063,0.7505,0.8610,0.0005


**Interpretasi**:
- `jumlah_penduduk` memiliki CV sangat tinggi (~164%) → distribusi sangat tidak merata
- `skewness > 2` pada penduduk & kepadatan → distribusi right-skewed kuat (dikuasai beberapa provinsi besar)
- Shapiro-Wilk `p < 0.05` pada hampir semua variabel → distribusi tidak normal, analisis nonparametrik lebih tepat

## 2. Koefisien Gini — Ketimpangan Distribusi Penduduk

In [3]:
def gini(arr):
    """Hitung koefisien Gini (0 = merata sempurna, 1 = tidak merata total)"""
    a = np.sort(arr)
    n = len(a)
    return (2 * np.sum((np.arange(1, n+1) * a))) / (n * np.sum(a)) - (n+1)/n

g = gini(df['jumlah_penduduk_ribu_2026'].values)
print(f'Koefisien Gini distribusi penduduk antar provinsi: {g:.4f}')
print()
print('Interpretasi:')
print('  0.00 – 0.30  : Rendah (relatif merata)')
print('  0.30 – 0.50  : Sedang')
print('  0.50 – 1.00  : Tinggi (sangat tidak merata)')
print(f'  → Nilai {g:.3f} menunjukkan ketimpangan TINGGI')

# Konsentrasi Jawa
jawa_mask = df['pulau'] == 'Jawa'
jawa_pct = df[jawa_mask]['distribusi_pct_2026'].sum()
print(f'\nKonsentrasi penduduk Pulau Jawa : {jawa_pct:.1f}% (6 dari 38 provinsi)')

Koefisien Gini distribusi penduduk antar provinsi: 0.6018

Interpretasi:
  0.00 – 0.30  : Rendah (relatif merata)
  0.30 – 0.50  : Sedang
  0.50 – 1.00  : Tinggi (sangat tidak merata)
  → Nilai 0.602 menunjukkan ketimpangan TINGGI

Konsentrasi penduduk Pulau Jawa : 55.4% (6 dari 38 provinsi)


## 3. Tren Laju Pertumbuhan Historis

In [4]:
periode_cols = [
    ('1971–1980', 'laju_1971_1980'),
    ('1980–1990', 'laju_1980_1990'),
    ('1990–2000', 'laju_1990_2000'),
    ('2000–2010', 'laju_2000_2010'),
    ('2010–2020', 'laju_2010_2020'),
    ('2020–2024', 'laju_2020_2024'),
]

print('=== TREN LAJU PERTUMBUHAN (median nasional, %/tahun) ===')
trend = []
for label, col in periode_cols:
    vals = df[col].dropna()
    trend.append({'periode': label, 'median': vals.median(), 'mean': vals.mean(), 'n': len(vals)})
    
trend_df = pd.DataFrame(trend)
print(trend_df.to_string(index=False))
print()
print('→ Tren umum MENURUN — perlambatan transisi demografi')
print('→ Anomali kenaikan 2000–2010: otonomi daerah & dekonsentrasi penduduk')

=== TREN LAJU PERTUMBUHAN (median nasional, %/tahun) ===
  periode  median   mean  n
1971–1980  2.6650 2.9000 26
1980–1990  2.6100 2.5504 26
1990–2000  1.5400 1.7697 30
2000–2010  1.9900 2.1736 33
2010–2020  1.3700 1.5656 32
2020–2024  1.3650 1.2958 38

→ Tren umum MENURUN — perlambatan transisi demografi
→ Anomali kenaikan 2000–2010: otonomi daerah & dekonsentrasi penduduk


## 4. Uji-T: Laju Pertumbuhan Jawa vs Luar Jawa

In [5]:
jawa_laju = df[df['pulau'] == 'Jawa']['laju_2020_2024'].dropna()
luar_laju = df[df['pulau'] != 'Jawa']['laju_2020_2024'].dropna()

# Levene test untuk kesamaan varians
levene_stat, levene_p = stats.levene(jawa_laju, luar_laju)
equal_var = levene_p > 0.05

# Independent t-test
t_stat, t_p = stats.ttest_ind(jawa_laju, luar_laju, equal_var=equal_var)

# Effect size (Cohen's d)
pooled_std = np.sqrt((jawa_laju.std()**2 + luar_laju.std()**2) / 2)
cohens_d = (jawa_laju.mean() - luar_laju.mean()) / pooled_std

print('=== UJI-T INDEPENDEN: Laju Pertumbuhan Jawa vs Luar Jawa (2020–2024) ===')
print(f'Levene test         : stat={levene_stat:.4f}, p={levene_p:.4f} → equal_var={equal_var}')
print(f'Jawa    (n={len(jawa_laju)})   : mean={jawa_laju.mean():.4f}%, std={jawa_laju.std():.4f}')
print(f'Luar Jawa (n={len(luar_laju)}) : mean={luar_laju.mean():.4f}%, std={luar_laju.std():.4f}')
print(f't-statistic         : {t_stat:.4f}')
print(f'p-value             : {t_p:.6f}')
print(f"Cohen's d           : {cohens_d:.4f}")
print()
sig = 'SIGNIFIKAN' if t_p < 0.05 else 'TIDAK SIGNIFIKAN'
print(f'→ Kesimpulan: Perbedaan {sig} (α = 0.05)')
print('→ Luar Jawa tumbuh lebih cepat, konsisten dengan redistribusi penduduk & DOB')

=== UJI-T INDEPENDEN: Laju Pertumbuhan Jawa vs Luar Jawa (2020–2024) ===
Levene test         : stat=1.7777, p=0.1908 → equal_var=True
Jawa    (n=6)   : mean=0.8317%, std=0.3267
Luar Jawa (n=32) : mean=1.3828%, std=0.2366
t-statistic         : -4.9350
p-value             : 0.000018
Cohen's d           : -1.9323

→ Kesimpulan: Perbedaan SIGNIFIKAN (α = 0.05)
→ Luar Jawa tumbuh lebih cepat, konsisten dengan redistribusi penduduk & DOB


## 5. Analisis Korelasi Antar Variabel Demografis

In [6]:
corr_cols = [
    'kepadatan_per_km2_2026', 'laju_2020_2024',
    'imr_per1000_2020', 'cbr_per1000_2020', 'tfr_2020', 'u5mr_per1000_2020'
]

corr_df = df[corr_cols].dropna()

# Pearson
print('=== MATRIKS KORELASI PEARSON ===')
print(corr_df.corr().round(3))

print()
# Spearman (lebih robust untuk distribusi skewed)
print('=== MATRIKS KORELASI SPEARMAN (lebih robust) ===')
print(corr_df.corr(method='spearman').round(3))

print()
print('Temuan utama:')
print('  CBR ↔ TFR    : r ≈ 0.96 (sangat kuat) — angka kelahiran kasar erat dengan total fertilitas')
print('  IMR ↔ TFR    : r ≈ 0.81 (kuat) — wilayah IMR tinggi = TFR tinggi (transisi demografi)')
print('  Kepadatan ↔ TFR : r ≈ -0.51 (sedang-negatif) — provinsi padat cenderung TFR lebih rendah')

=== MATRIKS KORELASI PEARSON ===
                        kepadatan_per_km2_2026  laju_2020_2024  \
kepadatan_per_km2_2026                  1.0000         -0.5870   
laju_2020_2024                         -0.5870          1.0000   
imr_per1000_2020                       -0.3100          0.5180   
cbr_per1000_2020                       -0.4470          0.8300   
tfr_2020                               -0.5080          0.7740   
u5mr_per1000_2020                      -0.2950          0.4980   

                        imr_per1000_2020  cbr_per1000_2020  tfr_2020  \
kepadatan_per_km2_2026           -0.3100           -0.4470   -0.5080   
laju_2020_2024                    0.5180            0.8300    0.7740   
imr_per1000_2020                  1.0000            0.8080    0.8050   
cbr_per1000_2020                  0.8080            1.0000    0.9610   
tfr_2020                          0.8050            0.9610    1.0000   
u5mr_per1000_2020                 0.9990            0.7940    0.7900   


## 6. Ranking Provinsi — Top & Bottom Performers

In [7]:
print('=== TOP 5 PERTUMBUHAN TERTINGGI (2020-2024) ===')
print(df.nlargest(5, 'laju_2020_2024')[['provinsi','laju_2020_2024','pulau']].to_string(index=False))

print()
print('=== TOP 5 PERTUMBUHAN TERENDAH (2020-2024) ===')
print(df.nsmallest(5, 'laju_2020_2024')[['provinsi','laju_2020_2024','pulau']].to_string(index=False))

print()
print('=== IMR TERBAIK (terendah) ===')
print(df.nsmallest(5, 'imr_per1000_2020')[['provinsi','imr_per1000_2020','tfr_2020']].to_string(index=False))

print()
print('=== IMR TERBURUK (tertinggi) ===')
print(df.nlargest(5, 'imr_per1000_2020')[['provinsi','imr_per1000_2020','tfr_2020']].to_string(index=False))

=== TOP 5 PERTUMBUHAN TERTINGGI (2020-2024) ===
           provinsi  laju_2020_2024         pulau
   KALIMANTAN TIMUR          1.9300    Kalimantan
        PAPUA BARAT          1.7100 Papua/NTT/NTB
  SULAWESI TENGGARA          1.6700      Sulawesi
NUSA TENGGARA TIMUR          1.6200 Papua/NTT/NTB
NUSA TENGGARA BARAT          1.6000 Papua/NTT/NTB

=== TOP 5 PERTUMBUHAN TERENDAH (2020-2024) ===
      provinsi  laju_2020_2024    pulau
   DKI JAKARTA          0.3100     Jawa
 DI YOGYAKARTA          0.6500     Jawa
          BALI          0.7100  Lainnya
    JAWA TIMUR          0.7500     Jawa
SULAWESI UTARA          0.8000 Sulawesi

=== IMR TERBAIK (terendah) ===
      provinsi  imr_per1000_2020  tfr_2020
   DKI JAKARTA           10.3800    1.7500
 DI YOGYAKARTA           10.9000    1.8900
   JAWA TENGAH           12.7700    2.0900
          BALI           13.2600    2.0400
KEPULAUAN RIAU           13.3100    2.2100

=== IMR TERBURUK (tertinggi) ===
      provinsi  imr_per1000_2020  tfr_20

## 7. Analisis Ketimpangan Per Pulau

In [8]:
pulau_stats = df.groupby('pulau').agg(
    n_provinsi=('provinsi', 'count'),
    total_penduduk_juta=('jumlah_penduduk_ribu_2026', lambda x: x.sum()/1000),
    distribusi_pct=('distribusi_pct_2026', 'sum'),
    median_laju=('laju_2020_2024', 'median'),
    median_imr=('imr_per1000_2020', 'median'),
    median_tfr=('tfr_2020', 'median')
).round(2).sort_values('total_penduduk_juta', ascending=False)

print('=== RINGKASAN PER PULAU ===')
print(pulau_stats.to_string())

=== RINGKASAN PER PULAU ===
               n_provinsi  total_penduduk_juta  distribusi_pct  median_laju  median_imr  median_tfr
pulau                                                                                              
Jawa                    6             159.1900         55.4300       0.8700     13.1300      2.0000
Sumatera               10              62.9900         21.9300       1.3700     16.7800      2.2800
Sulawesi                6              21.2800          7.4000       1.2300     25.5000      2.3100
Kalimantan              5              18.3200          6.3700       1.3700     17.2200      2.3100
Papua/NTT/NTB           8              17.5400          6.1000       1.5600     31.3600      2.7100
Lainnya                 3               7.8800          2.7300       1.3700     28.6100      2.4700
